This notebook allows to find cmip6 data with a combination of scenario / experiment with the required variables for the CATHERINA module. This is done by selecting the desired scenario / experiment, and finding the sources that provided all the data with the same initial conditions. 

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)

In [ ]:
# list of all available cmip6 data
path_list_scenario = "data/pangeo-cmip6.csv"
scenario = pd.read_csv(path_list_scenario)

In [ ]:
### List of desired experiments, to modify to run on other experiments
experiments = ["historical", "ssp370", "ssp585"]


# Filters to apply
activities = ["CMIP", "ScenarioMIP"]
tables = ["Amon", "Omon"]
variables = ["hurs", "psl", "ta", "tos"]

scenario_filter = scenario[
    scenario["activity_id"].isin(activities) &
    scenario["experiment_id"].isin(experiments) &
    scenario["table_id"].isin(tables) &
    scenario["variable_id"].isin(variables) &
    (scenario["grid_label"] == "gn")
]

In [ ]:
# Number of different experiments with the same source_id and member_id
scenario_filter["Nb_source_member_diff"] = scenario_filter.groupby(
    ["source_id", "member_id"]
)["activity_id"].transform(lambda x: x.size)

/tmp/ipykernel_26057/310192825.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  scenario_filter["Nb_source_member_diff"] = scenario_filter.groupby(["source_id", "member_id"])["activity_id"].transform(lambda x: x.size)


In [ ]:
# We filter on scenarios and initial conditions that are present for all experiments
scenario_full = scenario_filter[
    scenario_filter["Nb_source_member_diff"] == len(experiments) * len(variables)
].reset_index(drop=True).drop(columns=["Nb_source_member_diff"])

In [18]:
scenario_full.to_excel("data/list_of_cmip6_data_available.xlsx", index=False)